In [4]:
import requests
import pandas as pd
import os
from dotenv import load_dotenv
load_dotenv()
API_KEY = os.getenv("API_KEY")
URL = "https://api.gamebrain.co/v1/games?query=medieval+strategy+games"


def fetch_data():
    headers = {"x-api-key": API_KEY}
    response = requests.get(URL, headers=headers)
    response.raise_for_status()
    return response.json()


def transform(data):
    results = data["results"]

    rows = []
    for game in results:
        rows.append({
            "game_id": game.get("id"),
            "name": game.get("name"),
            "year": game.get("year"),
            "genre": game.get("genre"),
            "rating_mean": game.get("rating", {}).get("mean"),
            "rating_count": game.get("rating", {}).get("count"),
            "adult_only": game.get("adult_only"),
            "query": data.get("query"),
        })

    df = pd.DataFrame(rows)

    # ✅ Correct ingestion timestamp
    df["ingestion_date"] = pd.Timestamp.utcnow()

    # ✅ Dynamic partitions
    df["year_partition"] = df["ingestion_date"].dt.year
    df["month_partition"] = df["ingestion_date"].dt.month

    # ✅ Arrow-safe types
    df = df.copy()
    df = df.convert_dtypes()
    df["ingestion_date"] = pd.to_datetime(df["ingestion_date"]).dt.tz_localize(None)

    return df


def save_parquet(df):
    year = df["year_partition"].iloc[0]
    month = df["month_partition"].iloc[0]

    base_path = f"data/games/year={year}/month={month:02d}"
    os.makedirs(base_path, exist_ok=True)

    file_path = f"{base_path}/games_{year}_{month:02d}.parquet"

    df.to_parquet(file_path, engine="pyarrow", index=False)

    print(f"Saved: {file_path}")


if __name__ == "__main__":
    data = fetch_data()
    df = transform(data)
    save_parquet(df)

Saved: data/games/year=2026/month=04/games_2026_04.parquet


C:\Users\felha\AppData\Local\Temp\ipykernel_17128\79310592.py:36: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  df["ingestion_date"] = pd.Timestamp.utcnow()
